# Structural Patterns

## Adapter

In [1]:
import json
import re
from abc import ABC, abstractmethod


# Interface expected by new system
class ReportGeneratorInterface(ABC):
    @abstractmethod
    def generate(self, data: dict) -> str:
        """Returns JSON format"""
        pass


# Legacy system (can't modify)
class LegacyReportGenerator:
    def generate_report(self, data: dict) -> str:
        """Returns XML format"""
        xml = "<report>\n"
        for key, value in data.items():
            xml += f"  <{key}>{value}</{key}>\n"
        xml += "</report>"
        return xml


# Adapter
class LegacyReportAdapter(ReportGeneratorInterface):
    def __init__(self, legacy: LegacyReportGenerator):
        self._legacy = legacy

    def generate(self, data: dict) -> str:
        # Get XML from legacy
        xml = self._legacy.generate_report(data)
        # Convert to JSON
        return self._xml_to_json(xml)

    def _xml_to_json(self, xml: str) -> str:
        """Simple XML to JSON conversion"""
        result = {}
        pattern = r"<(\w+)>(.*?)</\1>"
        matches = re.findall(pattern, xml)
        for key, value in matches:
            # Try to convert numbers
            try:
                result[key] = float(value) if "." in value else int(value)
            except ValueError:
                result[key] = value
        return json.dumps(result)


# Dashboard (unchanged)
class AnalyticsDashboard:
    def display(self, json_data: str):
        data = json.loads(json_data)
        print("=== Analytics Dashboard ===")
        for key, value in data.items():
            print(f"  {key}: {value}")


# Usage
if __name__ == "__main__":
    # Create adapter wrapping legacy system
    legacy = LegacyReportGenerator()
    adapter = LegacyReportAdapter(legacy)

    # Dashboard works with adapter transparently
    dashboard = AnalyticsDashboard()

    # Sales report
    sales_data = {"total_sales": 150000, "orders": 1234, "avg_order": 121.55}
    json_report = adapter.generate(sales_data)
    dashboard.display(json_report)

    print()

    # Inventory report
    inventory_data = {"total_items": 5000, "low_stock": 45, "out_of_stock": 12}
    json_report = adapter.generate(inventory_data)
    dashboard.display(json_report)

=== Analytics Dashboard ===
  total_sales: 150000
  orders: 1234
  avg_order: 121.55

=== Analytics Dashboard ===
  total_items: 5000
  low_stock: 45
  out_of_stock: 12


Decorator

In [2]:
from abc import ABC, abstractmethod


# Interface
class OrderComponent(ABC):
    @abstractmethod
    def get_cost(self) -> float:
        pass

    @abstractmethod
    def get_description(self) -> str:
        pass


# Base order
class BaseOrder(OrderComponent):
    def __init__(self, base_price: float):
        self._base_price = base_price

    def get_cost(self) -> float:
        return self._base_price

    def get_description(self) -> str:
        return f"Base order: {self._base_price}€"


# Abstract decorator
class OrderDecorator(OrderComponent):
    def __init__(self, order: OrderComponent):
        self._order = order

    def get_cost(self) -> float:
        return self._order.get_cost()

    def get_description(self) -> str:
        return self._order.get_description()


# Concrete decorators
class ExpressShippingDecorator(OrderDecorator):
    def get_cost(self) -> float:
        return self._order.get_cost() + 15.00

    def get_description(self) -> str:
        return self._order.get_description() + "\n+ Express shipping: 15.00€"


class InsuranceDecorator(OrderDecorator):
    def get_cost(self) -> float:
        return self._order.get_cost() * 1.05  # +5%

    def get_description(self) -> str:
        insurance_cost = self._order.get_cost() * 0.05
        return self._order.get_description() + f"\n+ Insurance (5%): {insurance_cost:.2f}€"


class GiftWrapDecorator(OrderDecorator):
    def get_cost(self) -> float:
        return self._order.get_cost() + 5.00

    def get_description(self) -> str:
        return self._order.get_description() + "\n+ Gift wrap: 5.00€"


class DiscountDecorator(OrderDecorator):
    def __init__(self, order: OrderComponent, percent: float):
        super().__init__(order)
        self._percent = percent

    def get_cost(self) -> float:
        return self._order.get_cost() * (1 - self._percent / 100)

    def get_description(self) -> str:
        discount_amount = self._order.get_cost() * (self._percent / 100)
        return self._order.get_description() + f"\n- Discount ({self._percent}%): -{discount_amount:.2f}€"


class PremiumMemberDecorator(OrderDecorator):
    def get_cost(self) -> float:
        return self._order.get_cost() * 0.90  # -10%

    def get_description(self) -> str:
        discount_amount = self._order.get_cost() * 0.10
        return self._order.get_description() + f"\n- Premium member (10%): -{discount_amount:.2f}€"


# Usage
if __name__ == "__main__":
    # Simple order
    print("=== Simple Order ===")
    order = BaseOrder(100.00)
    print(order.get_description())
    print(f"TOTAL: {order.get_cost():.2f}€")

    print("\n=== Complex Order ===")
    # Stack decorators
    order = BaseOrder(100.00)
    order = ExpressShippingDecorator(order)
    order = InsuranceDecorator(order)
    order = GiftWrapDecorator(order)
    order = DiscountDecorator(order, percent=15)
    order = PremiumMemberDecorator(order)

    print(order.get_description())
    print(f"TOTAL: {order.get_cost():.2f}€")

=== Simple Order ===
Base order: 100.0€
TOTAL: 100.00€

=== Complex Order ===
Base order: 100.0€
+ Express shipping: 15.00€
+ Insurance (5%): 5.75€
+ Gift wrap: 5.00€
- Discount (15%): -18.86€
- Premium member (10%): -10.69€
TOTAL: 96.20€
